In [40]:
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
from nltk.sentiment.vader import SentimentIntensityAnalyzer

**Ekstrak Data**

In [41]:
# Fungsi untuk Ekstrak Data
def fetch_data_from_sql():
    conn_str = (
        "Driver={SQL Server};"                           # Driver untuk sql server
        "Server=DESKTOP-3QCNPJH\SQLEXPRESS01;"           # Nama server
        "Database=PortfolioProject_MarketingAnalytics;"  # Database
        "Trusted_Connection=yes;"                        # Menggunakan Windows Auntethification
    )
    
    conn  = pyodbc.connect(conn_str)
    query = "SELECT ReviewID, CustomerID, ProductID, ReviewDate, Rating, ReviewText FROM dbo.customer_reviews"
    
    df = pd.read_sql(query, conn)
    
    conn.close()
    
    return df

# Ekstrak Data
df = fetch_data_from_sql()
df.head()

<>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_19932\2788578113.py:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  "Server=DESKTOP-3QCNPJH\SQLEXPRESS01;"           # Nama server
C:\Users\user\AppData\Local\Temp\ipykernel_19932\2788578113.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


**Data Cleaning**

In [42]:
df['ReviewText'] = df['ReviewText'].str.strip().str.lower()
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"average experience, nothing special."
1,2,80,19,2024-12-25,5,the quality is top-notch.
2,3,50,13,2025-01-26,4,five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"average experience, nothing special."


**Menghitung Skor Sentimen**

In [43]:
sia = SentimentIntensityAnalyzer()

def calculate_sentiment(review):
    sentiment = sia.polarity_scores(review)
    return sentiment['compound']

df['SentimentScore'] = df['ReviewText'].apply(calculate_sentiment)
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore
0,1,77,18,2023-12-23,3,"average experience, nothing special.",-0.3089
1,2,80,19,2024-12-25,5,the quality is top-notch.,0.0000
2,3,50,13,2025-01-26,4,five stars for the quick delivery.,0.0000
3,4,78,15,2025-04-21,3,"good quality, but could be cheaper.",0.2382
4,5,64,2,2023-07-16,3,"average experience, nothing special.",-0.3089


**Pengkategorian Kombinasi Rating dan Score Sentimen**

In [47]:
def categorize_sentiment(score, rating):
    # Use both the text sentiment score and the numerical rating to determine sentiment category
    
    if score > 0.05:                 # Positive sentiment score
        if rating >= 4:
            return 'Positive'        # High rating and positive sentiment
        elif rating == 3:
            return 'Mixed Positive'  # Neutral rating but positive sentiment
        else:
            return 'Mixed Negative'  # Low rating but positive sentiment
    
    elif score < -0.05:              # Negative sentiment score
        if rating <= 2:
            return 'Negative'        # Low rating and negative sentiment
        elif rating == 3:
            return 'Mixed Negative'  # Neutral rating but negative sentiment
        else:
            return 'Mixed Positive'  # High rating but negative sentiment
    
    else:                            # Neutral sentiment score
        if rating >= 4:
            return 'Positive'        # High rating with neutral sentiment
        elif rating <= 2:
            return 'Negative'        # Low rating with neutral sentiment
        else:
            return 'Neutral'         # Neutral rating and neutral sentiment

df['SentimentCategory'] = df.apply(lambda x : categorize_sentiment(x['SentimentScore'], 
                                                                   x['Rating']
                                                                   ), axis = 1)
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,SentimentCategory
0,1,77,18,2023-12-23,3,"average experience, nothing special.",-0.3089,Mixed Negative
1,2,80,19,2024-12-25,5,the quality is top-notch.,0.0000,Positive
2,3,50,13,2025-01-26,4,five stars for the quick delivery.,0.0000,Positive
3,4,78,15,2025-04-21,3,"good quality, but could be cheaper.",0.2382,Mixed Positive
4,5,64,2,2023-07-16,3,"average experience, nothing special.",-0.3089,Mixed Negative


**Pembagian Range Skor Sentimen**

In [51]:
# Define a function to bucket sentiment scores into text ranges
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'   # Strongly positive sentiment
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'  # Mildly positive sentiment
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  # Mildly negative sentiment
    else:
        return '-1.0 to -0.5'  # Strongly negative sentiment

df['SentimentBucket'] = df['SentimentScore'].apply(sentiment_bucket)
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,SentimentCategory,SentimentBucket
0,1,77,18,2023-12-23,3,"average experience, nothing special.",-0.3089,Mixed Negative,-0.49 to 0.0
1,2,80,19,2024-12-25,5,the quality is top-notch.,0.0000,Positive,0.0 to 0.49
2,3,50,13,2025-01-26,4,five stars for the quick delivery.,0.0000,Positive,0.0 to 0.49
3,4,78,15,2025-04-21,3,"good quality, but could be cheaper.",0.2382,Mixed Positive,0.0 to 0.49
4,5,64,2,2023-07-16,3,"average experience, nothing special.",-0.3089,Mixed Negative,-0.49 to 0.0


In [52]:
df.to_csv('fact_customer_reviews_with_sentiment.csv', index=False)